# CareAssist - Placement Stability Prediction Model
**Train on real AFCARS FY2019-2024 data (all 6 years)**

## Instructions
1. Run Cell 1 to install packages
2. Run Cell 2 (imports)
3. Run Cell 3 - it will prompt you to **upload your 6 zip files**
4. Then **Runtime > Run all** remaining cells
5. Cell 16 will auto-download a zip with all outputs

In [ ]:
# Cell 1: Install packages
!pip install -q xgboost shap imbalanced-learn

In [ ]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, os, zipfile, glob
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)
import xgboost as xgb
import shap
import joblib
from imblearn.over_sampling import SMOTE

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
print('All imports OK')

In [ ]:
# Cell 3: Upload ALL 6 AFCARS zip files
from google.colab import files
print('Upload ALL 6 AFCARS zip files:')
print('  DS299 FC2019ABv1.zip')
print('  DS300 FC2020ABv1.zip')
print('  DS301 FC2021ABv1.zip')
print('  DS302 FC2022ABv1.zip')
print('  DS303 FC2023ABv1.zip')
print('  DS310 FC2024ABv1.zip')
print()
uploaded = files.upload()
print(f'Uploaded {len(uploaded)} files')

In [ ]:
# Cell 4: Extract zips and load all .tab files
for zf in glob.glob('*.zip'):
    print(f'Extracting {zf}...')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall('afcars_raw')

tab_files = glob.glob('afcars_raw/**/*.tab', recursive=True)
print(f'Found {len(tab_files)} .tab files:')
for f in sorted(tab_files):
    print(f'  {f}')

USE_COLS = [
    'RecNumbr', 'FIPSCode', 'StFCID', 'FY',
    'NUMPLEP', 'TOTALREM', 'CURPLSET', 'CASEGOAL', 'SEX', 'CLINDIS',
    'MR', 'VISHEAR', 'PHYDIS', 'EmotDist', 'OTHERMED', 'CHBEHPRB',
    'PHYABUSE', 'SEXABUSE', 'NEGLECT', 'AAPARENT', 'DAPARENT',
    'AACHILD', 'DACHILD', 'CHILDIS', 'PRTSDIED', 'PRTSJAIL',
    'NOCOPE', 'ABANDMNT', 'RELINQSH', 'HOUSING', 'MANREM',
    'EVERADPT', 'DISREASN', 'PLACEOUT',
    'AgeAtLatRem', 'RaceEthn', 'SettingLOS', 'LatRemLOS',
]

def load_tab(path):
    header = pd.read_csv(path, sep='\t', nrows=0)
    cols = [c for c in USE_COLS if c in header.columns]
    d = pd.read_csv(path, sep='\t', usecols=cols, dtype=str, low_memory=False)
    for c in d.columns:
        if c not in ('RecNumbr', 'FIPSCode', 'StFCID', 'FY'):
            d[c] = pd.to_numeric(d[c], errors='coerce')
    return d

dfs = []
for f in sorted(tab_files):
    print(f'Loading {os.path.basename(f)}...')
    d = load_tab(f)
    print(f'  {len(d):,} records')
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
print(f'\nCombined: {len(df):,} records x {df.shape[1]} columns')
print(f'Fiscal years: {sorted(df["FY"].dropna().unique())}')
df.head()

In [ ]:
# Cell 5: Define target variable
# Disrupted = discharge reason Transfer(6)/Runaway(7) OR 3+ placements
# NUMPLEP excluded from features to avoid data leakage

DISRUPTION_CODES = {6, 7}

def label_disruption(row):
    d = row['DISREASN']
    n = row['NUMPLEP']
    if pd.notna(d) and int(d) in DISRUPTION_CODES:
        return 1
    if pd.notna(n) and int(n) >= 3:
        return 1
    return 0

df['disruption'] = df.apply(label_disruption, axis=1)
pos = df['disruption'].sum()
print(f'Disrupted: {pos:,} ({100*pos/len(df):.1f}%)')
print(f'Stable:    {len(df)-pos:,} ({100*(len(df)-pos)/len(df):.1f}%)')
print('\nBy fiscal year:')
print(df.groupby('FY')['disruption'].agg(['count','mean','sum']).to_string())

In [ ]:
# Cell 6: Feature engineering
df['age_at_removal'] = df['AgeAtLatRem'].where(df['AgeAtLatRem'] < 99)

disability_cols = ['MR', 'VISHEAR', 'PHYDIS', 'EmotDist', 'OTHERMED']
for c in disability_cols:
    df[c] = df[c].fillna(0).clip(0, 1).astype(int)
df['has_disability'] = df[disability_cols].max(axis=1)
df['has_clinical_disability'] = (df['CLINDIS'] == 1).astype(int)
df['has_behavioral'] = df['CHBEHPRB'].fillna(0).clip(0, 1).astype(int)

removal_reason_cols = [
    'PHYABUSE','SEXABUSE','NEGLECT','AAPARENT','DAPARENT',
    'AACHILD','DACHILD','CHILDIS','PRTSDIED','PRTSJAIL',
    'NOCOPE','ABANDMNT','RELINQSH','HOUSING'
]
for c in removal_reason_cols:
    df[c] = df[c].fillna(0).clip(0, 1).astype(int)
df['num_removal_reasons'] = df[removal_reason_cols].sum(axis=1)

df['placement_type'] = df['CURPLSET'].where(df['CURPLSET'].isin([1,2,3,4,5,6,7,8]))
df['case_goal'] = df['CASEGOAL'].where(df['CASEGOAL'].isin([1,2,3,4,5,6,7]))
df['is_male'] = (df['SEX'] == 1).astype(int)
df['race'] = df['RaceEthn'].where(df['RaceEthn'].isin(range(1, 8)))
df['total_removals'] = df['TOTALREM'].where(df['TOTALREM'] < 98)

if 'SettingLOS' in df.columns:
    df['los_current_setting'] = pd.to_numeric(df['SettingLOS'], errors='coerce')
else:
    df['los_current_setting'] = np.nan
if 'LatRemLOS' in df.columns:
    df['los_latest_removal'] = pd.to_numeric(df['LatRemLOS'], errors='coerce')
else:
    df['los_latest_removal'] = np.nan

df['ever_adopted'] = df['EVERADPT'].fillna(0).clip(0, 1).astype(int)
df['mandatory_removal'] = df['MANREM'].fillna(0).clip(0, 1).astype(int)

FEATURE_COLS = [
    'age_at_removal','is_male','race',
    'total_removals','placement_type','case_goal',
    'has_disability','has_clinical_disability','has_behavioral',
    'num_removal_reasons',
    'PHYABUSE','SEXABUSE','NEGLECT','AAPARENT','DAPARENT',
    'NOCOPE','ABANDMNT','HOUSING',
    'los_current_setting','los_latest_removal',
    'ever_adopted','mandatory_removal',
]
print(f'{len(FEATURE_COLS)} features (NUMPLEP excluded to prevent leakage)')

In [ ]:
# Cell 7: EDA
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

df['disruption'].value_counts().plot(kind='bar', ax=axes[0,0], color=['steelblue','coral'])
axes[0,0].set_title('Target Distribution')
axes[0,0].set_xticklabels(['Stable','Disrupted'], rotation=0)

df[df['disruption']==0]['age_at_removal'].hist(bins=20, ax=axes[0,1], alpha=0.6, label='Stable', color='steelblue')
df[df['disruption']==1]['age_at_removal'].hist(bins=20, ax=axes[0,1], alpha=0.6, label='Disrupted', color='coral')
axes[0,1].set_title('Age at Removal'); axes[0,1].legend()

pt_labels = {1:'Pre-Adopt',2:'Foster-Rel',3:'Foster-NonRel',4:'Group Home',5:'Institution',6:'Sup IL',7:'Runaway',8:'Trial Home'}
pt_rates = df.groupby('placement_type')['disruption'].mean().sort_values(ascending=False)
pt_rates.index = [pt_labels.get(int(i),str(i)) for i in pt_rates.index]
pt_rates.plot(kind='barh', ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Disruption by Placement Type')

cg_labels = {1:'Reunify',2:'Relative',3:'Adoption',4:'Long-term FC',5:'Emancipation',6:'Guardianship',7:'Not established'}
cg_rates = df.groupby('case_goal')['disruption'].mean().sort_values(ascending=False)
cg_rates.index = [cg_labels.get(int(i),str(i)) for i in cg_rates.index]
cg_rates.plot(kind='barh', ax=axes[1,0], color='coral')
axes[1,0].set_title('Disruption by Case Goal')

tr = df.groupby('total_removals')['disruption'].mean().head(10)
tr.plot(kind='bar', ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Disruption by Total Removals')
axes[1,1].tick_params(axis='x', rotation=0)

yr = df.groupby('FY')['disruption'].mean()
yr.plot(kind='bar', ax=axes[1,2], color='coral')
axes[1,2].set_title('Disruption by Year')
axes[1,2].tick_params(axis='x', rotation=45)

plt.suptitle('CareAssist EDA (FY2019-2024)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8: Prepare data
model_df = df[FEATURE_COLS + ['disruption']].copy()
cat_cols = ['placement_type','case_goal','race']
model_df = pd.get_dummies(model_df, columns=cat_cols, prefix=cat_cols, dummy_na=False)

for c in model_df.columns:
    if model_df[c].isna().any():
        model_df[c] = model_df[c].fillna(model_df[c].median())

y = model_df['disruption']
X = model_df.drop(columns=['disruption'])
feature_names = list(X.columns)
print(f'Features: {len(feature_names)}, Records: {len(X):,}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'Disruption rate: train={y_train.mean():.3f} test={y_test.mean():.3f}')

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
print(f'After SMOTE: {len(X_train_sm):,} balanced samples')

In [ ]:
# Cell 9: Random Forest
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
rf_proba = rf.predict_proba(X_test)[:,1]
rf_auc = roc_auc_score(y_test, rf_proba)
rf_ap = average_precision_score(y_test, rf_proba)
print(f'  ROC-AUC: {rf_auc:.4f}  Avg Precision: {rf_ap:.4f}')

In [ ]:
# Cell 10: XGBoost
print('Training XGBoost...')
scale_pos = (y_train_sm==0).sum() / max((y_train_sm==1).sum(), 1)
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos,
    eval_metric='logloss', random_state=42, n_jobs=-1, use_label_encoder=False)
xgb_model.fit(X_train_sm, y_train_sm, verbose=False)
xgb_proba = xgb_model.predict_proba(X_test)[:,1]
xgb_auc = roc_auc_score(y_test, xgb_proba)
xgb_ap = average_precision_score(y_test, xgb_proba)
print(f'  ROC-AUC: {xgb_auc:.4f}  Avg Precision: {xgb_ap:.4f}')

if xgb_auc > rf_auc:
    best_model, best_proba, best_name, best_auc = xgb_model, xgb_proba, 'XGBoost', xgb_auc
else:
    best_model, best_proba, best_name, best_auc = rf, rf_proba, 'RandomForest', rf_auc
print(f'\nBest: {best_name} (AUC={best_auc:.4f})')

In [ ]:
# Cell 11: Evaluation plots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_xg, tpr_xg, _ = roc_curve(y_test, xgb_proba)
axes[0].plot(fpr_rf, tpr_rf, 'b-', lw=2, label=f'RF AUC={rf_auc:.3f}')
axes[0].plot(fpr_xg, tpr_xg, 'r-', lw=2, label=f'XGB AUC={xgb_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--',lw=0.5)
axes[0].set_title('ROC Curve'); axes[0].legend()

p_rf, r_rf, _ = precision_recall_curve(y_test, rf_proba)
p_xg, r_xg, _ = precision_recall_curve(y_test, xgb_proba)
axes[1].plot(r_rf, p_rf, 'b-', lw=2, label=f'RF AP={rf_ap:.3f}')
axes[1].plot(r_xg, p_xg, 'r-', lw=2, label=f'XGB AP={xgb_ap:.3f}')
axes[1].set_title('Precision-Recall'); axes[1].legend()

y_pred = (best_proba >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
    xticklabels=['Stable','Disrupted'], yticklabels=['Stable','Disrupted'], ax=axes[2])
axes[2].set_title(f'Confusion Matrix ({best_name})')

plt.suptitle('Model Evaluation', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print(classification_report(y_test, y_pred, target_names=['Stable','Disrupted']))

In [ ]:
# Cell 12: Feature importance
feat_imp = pd.DataFrame({'feature':feature_names, 'importance':best_model.feature_importances_}
    ).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top20 = feat_imp.head(20)
ax.barh(range(len(top20)), top20['importance'].values, color='steelblue')
ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20['feature'].values)
ax.invert_yaxis(); ax.set_title(f'Feature Importance ({best_name})')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(feat_imp.head(20).to_string(index=False))

In [ ]:
# Cell 13: SHAP
print('Computing SHAP (sample=1000)...')
explainer = shap.TreeExplainer(best_model)
X_sample = X_test.sample(min(1000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list): shap_values = shap_values[1]

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False, max_display=20)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

idx = np.argmax(best_model.predict_proba(X_sample)[:,1])
ev = explainer.expected_value
if isinstance(ev, list): ev = ev[1]
shap.plots.waterfall(shap.Explanation(
    values=shap_values[idx], base_values=ev,
    data=X_sample.iloc[idx], feature_names=feature_names), show=True)
print('SHAP done')

In [ ]:
# Cell 14: Score all records
print('Scoring all records...')
X_all = model_df.drop(columns=['disruption'])
df['priority_score'] = best_model.predict_proba(X_all)[:,1]
df['risk_tier'] = pd.cut(df['priority_score'],
    bins=[0,0.3,0.6,0.8,1.0], labels=['Low','Medium','High','Critical'], include_lowest=True)

print('Risk Tier Distribution:')
tier_dist = df['risk_tier'].value_counts()
for t in ['Critical','High','Medium','Low']:
    if t in tier_dist.index:
        print(f'  {t:>10}: {tier_dist[t]:>8,} ({100*tier_dist[t]/len(df):.1f}%)')

fig, ax = plt.subplots(figsize=(10,5))
ax.hist(df['priority_score'], bins=50, color='steelblue', edgecolor='white')
for v,c,l in [(0.3,'green','Low/Med'),(0.6,'orange','Med/High'),(0.8,'red','High/Crit')]:
    ax.axvline(v, color=c, ls='--', label=l)
ax.set_title('Priority Score Distribution'); ax.legend()
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 15: Export everything
os.makedirs('careassist_model_output', exist_ok=True)

joblib.dump(best_model, 'careassist_model_output/placement_model.pkl')
print('placement_model.pkl')

meta = {
    'model_type': best_name, 'roc_auc': round(best_auc,4),
    'rf_auc': round(rf_auc,4), 'xgb_auc': round(xgb_auc,4),
    'feature_names': feature_names, 'feature_count': len(feature_names),
    'train_samples': int(len(X_train_sm)), 'test_samples': int(len(X_test)),
    'total_records': int(len(df)), 'disruption_rate': round(float(y.mean()),4),
    'years': sorted([str(x) for x in df['FY'].dropna().unique()]),
    'risk_tiers': {'Low':'0-0.3','Medium':'0.3-0.6','High':'0.6-0.8','Critical':'0.8-1.0'},
}
with open('careassist_model_output/model_metadata.json','w') as f:
    json.dump(meta, f, indent=2)
print('model_metadata.json')

feat_imp.to_csv('careassist_model_output/feature_importance.csv', index=False)
print('feature_importance.csv')

export_cols = ['RecNumbr','StFCID','FIPSCode','FY',
    'age_at_removal','is_male','race','total_removals','placement_type','case_goal',
    'has_disability','has_clinical_disability','has_behavioral','num_removal_reasons',
    'los_current_setting','los_latest_removal','disruption','priority_score','risk_tier']
export_cols = [c for c in export_cols if c in df.columns]
df[export_cols].to_csv('careassist_model_output/scored_cases.csv', index=False)
print(f'scored_cases.csv ({len(df):,} records)')

import shutil
for f in ['eda_plots.png','model_evaluation.png','feature_importance.png','shap_summary.png','score_distribution.png']:
    if os.path.exists(f): shutil.copy(f, f'careassist_model_output/{f}')
print(f'\nAll files: {os.listdir("careassist_model_output")}')

In [ ]:
# Cell 16: Download zip
import shutil
shutil.make_archive('careassist_model_output', 'zip', '.', 'careassist_model_output')
print('Downloading careassist_model_output.zip...')
files.download('careassist_model_output.zip')

In [ ]:
# Cell 17: COPY THIS OUTPUT AND PASTE IT BACK IN VS CODE CHAT
print('='*60)
print('COPY EVERYTHING BELOW THIS LINE')
print('='*60)
print(f'MODEL_TYPE={best_name}')
print(f'ROC_AUC={best_auc:.4f}')
print(f'RF_AUC={rf_auc:.4f}')
print(f'XGB_AUC={xgb_auc:.4f}')
print(f'TOTAL_RECORDS={len(df):,}')
print(f'DISRUPTION_RATE={y.mean():.4f}')
print(f'FEATURES={len(feature_names)}')
print(f'TRAIN_SAMPLES={len(X_train_sm):,}')
print(f'TEST_SAMPLES={len(X_test):,}')
print(f'YEARS={sorted(df["FY"].dropna().unique().tolist())}')
print(f'\nTIER_DISTRIBUTION:')
for t in ['Critical','High','Medium','Low']:
    if t in tier_dist.index: print(f'  {t}: {tier_dist[t]:,}')
print(f'\nTOP_10_FEATURES:')
for _,r in feat_imp.head(10).iterrows():
    print(f'  {r["feature"]}: {r["importance"]:.4f}')
print(f'\nCLASSIFICATION_REPORT:')
print(classification_report(y_test, y_pred, target_names=['Stable','Disrupted']))
cm = confusion_matrix(y_test, y_pred)
print(f'CONFUSION_MATRIX: TN={cm[0,0]:,} FP={cm[0,1]:,} FN={cm[1,0]:,} TP={cm[1,1]:,}')
print('='*60)

# CareAssist — Placement Stability Prediction Model
**Train on real AFCARS FY2019-2024 data (all 6 years)**

## Instructions
1. Run Cell 1 to install packages
2. Run Cell 2 (imports)
3. Run Cell 3 — it will prompt you to **upload your 6 zip files**:
   - `DS299 FC2019ABv1.zip`
   - `DS300 FC2020ABv1.zip`
   - `DS301 FC2021ABv1.zip`
   - `DS302 FC2022ABv1.zip`
   - `DS303 FC2023ABv1.zip`
   - `DS310 FC2024ABv1.zip`
4. Then **Runtime > Run all** remaining cells
5. Cell 16 will auto-download a zip with all outputs

In [ ]:
# Cell 1: Install packages
!pip install -q xgboost shap imbalanced-learn

In [ ]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, os, zipfile, glob
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)
import xgboost as xgb
import shap
import joblib
from imblearn.over_sampling import SMOTE

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
print('All imports OK')

In [ ]:
# Cell 3: Upload ALL 6 AFCARS zip files
from google.colab import files
print('Upload ALL 6 AFCARS zip files:')
print('  DS299 FC2019ABv1.zip')
print('  DS300 FC2020ABv1.zip')
print('  DS301 FC2021ABv1.zip')
print('  DS302 FC2022ABv1.zip')
print('  DS303 FC2023ABv1.zip')
print('  DS310 FC2024ABv1.zip')
print()
uploaded = files.upload()
print(f'Uploaded {len(uploaded)} files')

In [ ]:
# Cell 4: Extract zips and load all .tab files
for zf in glob.glob('*.zip'):
    print(f'Extracting {zf}...')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall('afcars_raw')

tab_files = glob.glob('afcars_raw/**/*.tab', recursive=True)
print(f'Found {len(tab_files)} .tab files:')
for f in sorted(tab_files):
    print(f'  {f}')

USE_COLS = [
    'RecNumbr', 'FIPSCode', 'StFCID', 'FY',
    'NUMPLEP', 'TOTALREM', 'CURPLSET', 'CASEGOAL', 'SEX', 'CLINDIS',
    'MR', 'VISHEAR', 'PHYDIS', 'EmotDist', 'OTHERMED', 'CHBEHPRB',
    'PHYABUSE', 'SEXABUSE', 'NEGLECT', 'AAPARENT', 'DAPARENT',
    'AACHILD', 'DACHILD', 'CHILDIS', 'PRTSDIED', 'PRTSJAIL',
    'NOCOPE', 'ABANDMNT', 'RELINQSH', 'HOUSING', 'MANREM',
    'EVERADPT', 'DISREASN', 'PLACEOUT',
    'AgeAtLatRem', 'RaceEthn', 'SettingLOS', 'LatRemLOS',
]

def load_tab(path):
    header = pd.read_csv(path, sep='\t', nrows=0)
    cols = [c for c in USE_COLS if c in header.columns]
    d = pd.read_csv(path, sep='\t', usecols=cols, dtype=str, low_memory=False)
    for c in d.columns:
        if c not in ('RecNumbr', 'FIPSCode', 'StFCID', 'FY'):
            d[c] = pd.to_numeric(d[c], errors='coerce')
    return d

dfs = []
for f in sorted(tab_files):
    print(f'Loading {os.path.basename(f)}...')
    d = load_tab(f)
    print(f'  {len(d):,} records')
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True)
print(f'\nCombined: {len(df):,} records x {df.shape[1]} columns')
print(f'Fiscal years: {sorted(df["FY"].dropna().unique())}')
df.head()

In [ ]:
# Cell 5: Define target variable
# Disrupted = discharge reason Transfer(6)/Runaway(7) OR 3+ placements
# NUMPLEP excluded from features to avoid data leakage

DISRUPTION_CODES = {6, 7}

def label_disruption(row):
    d = row['DISREASN']
    n = row['NUMPLEP']
    if pd.notna(d) and int(d) in DISRUPTION_CODES:
        return 1
    if pd.notna(n) and int(n) >= 3:
        return 1
    return 0

df['disruption'] = df.apply(label_disruption, axis=1)
pos = df['disruption'].sum()
print(f'Disrupted: {pos:,} ({100*pos/len(df):.1f}%)')
print(f'Stable:    {len(df)-pos:,} ({100*(len(df)-pos)/len(df):.1f}%)')
print('\nBy fiscal year:')
print(df.groupby('FY')['disruption'].agg(['count','mean','sum']).to_string())

In [ ]:
# Cell 6: Feature engineering
df['age_at_removal'] = df['AgeAtLatRem'].where(df['AgeAtLatRem'] < 99)

disability_cols = ['MR', 'VISHEAR', 'PHYDIS', 'EmotDist', 'OTHERMED']
for c in disability_cols:
    df[c] = df[c].fillna(0).clip(0, 1).astype(int)
df['has_disability'] = df[disability_cols].max(axis=1)
df['has_clinical_disability'] = (df['CLINDIS'] == 1).astype(int)
df['has_behavioral'] = df['CHBEHPRB'].fillna(0).clip(0, 1).astype(int)

removal_reason_cols = [
    'PHYABUSE','SEXABUSE','NEGLECT','AAPARENT','DAPARENT',
    'AACHILD','DACHILD','CHILDIS','PRTSDIED','PRTSJAIL',
    'NOCOPE','ABANDMNT','RELINQSH','HOUSING'
]
for c in removal_reason_cols:
    df[c] = df[c].fillna(0).clip(0, 1).astype(int)
df['num_removal_reasons'] = df[removal_reason_cols].sum(axis=1)

df['placement_type'] = df['CURPLSET'].where(df['CURPLSET'].isin([1,2,3,4,5,6,7,8]))
df['case_goal'] = df['CASEGOAL'].where(df['CASEGOAL'].isin([1,2,3,4,5,6,7]))
df['is_male'] = (df['SEX'] == 1).astype(int)
df['race'] = df['RaceEthn'].where(df['RaceEthn'].isin(range(1, 8)))
df['total_removals'] = df['TOTALREM'].where(df['TOTALREM'] < 98)

if 'SettingLOS' in df.columns:
    df['los_current_setting'] = pd.to_numeric(df['SettingLOS'], errors='coerce')
else:
    df['los_current_setting'] = np.nan
if 'LatRemLOS' in df.columns:
    df['los_latest_removal'] = pd.to_numeric(df['LatRemLOS'], errors='coerce')
else:
    df['los_latest_removal'] = np.nan

df['ever_adopted'] = df['EVERADPT'].fillna(0).clip(0, 1).astype(int)
df['mandatory_removal'] = df['MANREM'].fillna(0).clip(0, 1).astype(int)

FEATURE_COLS = [
    'age_at_removal','is_male','race',
    'total_removals','placement_type','case_goal',
    'has_disability','has_clinical_disability','has_behavioral',
    'num_removal_reasons',
    'PHYABUSE','SEXABUSE','NEGLECT','AAPARENT','DAPARENT',
    'NOCOPE','ABANDMNT','HOUSING',
    'los_current_setting','los_latest_removal',
    'ever_adopted','mandatory_removal',
]
print(f'{len(FEATURE_COLS)} features (NUMPLEP excluded to prevent leakage)')

In [ ]:
# Cell 7: EDA
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

df['disruption'].value_counts().plot(kind='bar', ax=axes[0,0], color=['steelblue','coral'])
axes[0,0].set_title('Target Distribution')
axes[0,0].set_xticklabels(['Stable','Disrupted'], rotation=0)

df[df['disruption']==0]['age_at_removal'].hist(bins=20, ax=axes[0,1], alpha=0.6, label='Stable', color='steelblue')
df[df['disruption']==1]['age_at_removal'].hist(bins=20, ax=axes[0,1], alpha=0.6, label='Disrupted', color='coral')
axes[0,1].set_title('Age at Removal'); axes[0,1].legend()

pt_labels = {1:'Pre-Adopt',2:'Foster-Rel',3:'Foster-NonRel',4:'Group Home',5:'Institution',6:'Sup IL',7:'Runaway',8:'Trial Home'}
pt_rates = df.groupby('placement_type')['disruption'].mean().sort_values(ascending=False)
pt_rates.index = [pt_labels.get(int(i),str(i)) for i in pt_rates.index]
pt_rates.plot(kind='barh', ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Disruption by Placement Type')

cg_labels = {1:'Reunify',2:'Relative',3:'Adoption',4:'Long-term FC',5:'Emancipation',6:'Guardianship',7:'Not established'}
cg_rates = df.groupby('case_goal')['disruption'].mean().sort_values(ascending=False)
cg_rates.index = [cg_labels.get(int(i),str(i)) for i in cg_rates.index]
cg_rates.plot(kind='barh', ax=axes[1,0], color='coral')
axes[1,0].set_title('Disruption by Case Goal')

tr = df.groupby('total_removals')['disruption'].mean().head(10)
tr.plot(kind='bar', ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Disruption by Total Removals')
axes[1,1].tick_params(axis='x', rotation=0)

yr = df.groupby('FY')['disruption'].mean()
yr.plot(kind='bar', ax=axes[1,2], color='coral')
axes[1,2].set_title('Disruption by Year')
axes[1,2].tick_params(axis='x', rotation=45)

plt.suptitle('CareAssist EDA (FY2019-2024)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8: Prepare data
model_df = df[FEATURE_COLS + ['disruption']].copy()
cat_cols = ['placement_type','case_goal','race']
model_df = pd.get_dummies(model_df, columns=cat_cols, prefix=cat_cols, dummy_na=False)

for c in model_df.columns:
    if model_df[c].isna().any():
        model_df[c] = model_df[c].fillna(model_df[c].median())

y = model_df['disruption']
X = model_df.drop(columns=['disruption'])
feature_names = list(X.columns)
print(f'Features: {len(feature_names)}, Records: {len(X):,}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')
print(f'Disruption rate: train={y_train.mean():.3f} test={y_test.mean():.3f}')

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
print(f'After SMOTE: {len(X_train_sm):,} balanced samples')

In [ ]:
# Cell 9: Random Forest
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=300, max_depth=12, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
rf_proba = rf.predict_proba(X_test)[:,1]
rf_auc = roc_auc_score(y_test, rf_proba)
rf_ap = average_precision_score(y_test, rf_proba)
print(f'  ROC-AUC: {rf_auc:.4f}  Avg Precision: {rf_ap:.4f}')

In [ ]:
# Cell 10: XGBoost
print('Training XGBoost...')
scale_pos = (y_train_sm==0).sum() / max((y_train_sm==1).sum(), 1)
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos,
    eval_metric='logloss', random_state=42, n_jobs=-1, use_label_encoder=False)
xgb_model.fit(X_train_sm, y_train_sm, verbose=False)
xgb_proba = xgb_model.predict_proba(X_test)[:,1]
xgb_auc = roc_auc_score(y_test, xgb_proba)
xgb_ap = average_precision_score(y_test, xgb_proba)
print(f'  ROC-AUC: {xgb_auc:.4f}  Avg Precision: {xgb_ap:.4f}')

if xgb_auc > rf_auc:
    best_model, best_proba, best_name, best_auc = xgb_model, xgb_proba, 'XGBoost', xgb_auc
else:
    best_model, best_proba, best_name, best_auc = rf, rf_proba, 'RandomForest', rf_auc
print(f'\nBest: {best_name} (AUC={best_auc:.4f})')

In [ ]:
# Cell 11: Evaluation plots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_xg, tpr_xg, _ = roc_curve(y_test, xgb_proba)
axes[0].plot(fpr_rf, tpr_rf, 'b-', lw=2, label=f'RF AUC={rf_auc:.3f}')
axes[0].plot(fpr_xg, tpr_xg, 'r-', lw=2, label=f'XGB AUC={xgb_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--',lw=0.5)
axes[0].set_title('ROC Curve'); axes[0].legend()

p_rf, r_rf, _ = precision_recall_curve(y_test, rf_proba)
p_xg, r_xg, _ = precision_recall_curve(y_test, xgb_proba)
axes[1].plot(r_rf, p_rf, 'b-', lw=2, label=f'RF AP={rf_ap:.3f}')
axes[1].plot(r_xg, p_xg, 'r-', lw=2, label=f'XGB AP={xgb_ap:.3f}')
axes[1].set_title('Precision-Recall'); axes[1].legend()

y_pred = (best_proba >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
    xticklabels=['Stable','Disrupted'], yticklabels=['Stable','Disrupted'], ax=axes[2])
axes[2].set_title(f'Confusion Matrix ({best_name})')

plt.suptitle('Model Evaluation', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print(classification_report(y_test, y_pred, target_names=['Stable','Disrupted']))

In [ ]:
# Cell 12: Feature importance
feat_imp = pd.DataFrame({'feature':feature_names, 'importance':best_model.feature_importances_}
    ).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top20 = feat_imp.head(20)
ax.barh(range(len(top20)), top20['importance'].values, color='steelblue')
ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20['feature'].values)
ax.invert_yaxis(); ax.set_title(f'Feature Importance ({best_name})')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(feat_imp.head(20).to_string(index=False))

In [ ]:
# Cell 13: SHAP
print('Computing SHAP (sample=1000)...')
explainer = shap.TreeExplainer(best_model)
X_sample = X_test.sample(min(1000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list): shap_values = shap_values[1]

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False, max_display=20)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

idx = np.argmax(best_model.predict_proba(X_sample)[:,1])
ev = explainer.expected_value
if isinstance(ev, list): ev = ev[1]
shap.plots.waterfall(shap.Explanation(
    values=shap_values[idx], base_values=ev,
    data=X_sample.iloc[idx], feature_names=feature_names), show=True)
print('SHAP done')

In [ ]:
# Cell 14: Score all records
print('Scoring all records...')
X_all = model_df.drop(columns=['disruption'])
df['priority_score'] = best_model.predict_proba(X_all)[:,1]
df['risk_tier'] = pd.cut(df['priority_score'],
    bins=[0,0.3,0.6,0.8,1.0], labels=['Low','Medium','High','Critical'], include_lowest=True)

print('Risk Tier Distribution:')
tier_dist = df['risk_tier'].value_counts()
for t in ['Critical','High','Medium','Low']:
    if t in tier_dist.index:
        print(f'  {t:>10}: {tier_dist[t]:>8,} ({100*tier_dist[t]/len(df):.1f}%)')

fig, ax = plt.subplots(figsize=(10,5))
ax.hist(df['priority_score'], bins=50, color='steelblue', edgecolor='white')
for v,c,l in [(0.3,'green','Low/Med'),(0.6,'orange','Med/High'),(0.8,'red','High/Crit')]:
    ax.axvline(v, color=c, ls='--', label=l)
ax.set_title('Priority Score Distribution'); ax.legend()
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 15: Export everything
os.makedirs('careassist_model_output', exist_ok=True)

joblib.dump(best_model, 'careassist_model_output/placement_model.pkl')
print('placement_model.pkl')

meta = {
    'model_type': best_name, 'roc_auc': round(best_auc,4),
    'rf_auc': round(rf_auc,4), 'xgb_auc': round(xgb_auc,4),
    'feature_names': feature_names, 'feature_count': len(feature_names),
    'train_samples': int(len(X_train_sm)), 'test_samples': int(len(X_test)),
    'total_records': int(len(df)), 'disruption_rate': round(float(y.mean()),4),
    'years': sorted([str(x) for x in df['FY'].dropna().unique()]),
    'risk_tiers': {'Low':'0-0.3','Medium':'0.3-0.6','High':'0.6-0.8','Critical':'0.8-1.0'},
}
with open('careassist_model_output/model_metadata.json','w') as f:
    json.dump(meta, f, indent=2)
print('model_metadata.json')

feat_imp.to_csv('careassist_model_output/feature_importance.csv', index=False)
print('feature_importance.csv')

export_cols = ['RecNumbr','StFCID','FIPSCode','FY',
    'age_at_removal','is_male','race','total_removals','placement_type','case_goal',
    'has_disability','has_clinical_disability','has_behavioral','num_removal_reasons',
    'los_current_setting','los_latest_removal','disruption','priority_score','risk_tier']
export_cols = [c for c in export_cols if c in df.columns]
df[export_cols].to_csv('careassist_model_output/scored_cases.csv', index=False)
print(f'scored_cases.csv ({len(df):,} records)')

import shutil
for f in ['eda_plots.png','model_evaluation.png','feature_importance.png','shap_summary.png','score_distribution.png']:
    if os.path.exists(f): shutil.copy(f, f'careassist_model_output/{f}')
print(f'\nAll files: {os.listdir("careassist_model_output")}')

In [ ]:
# Cell 16: Download zip
import shutil
shutil.make_archive('careassist_model_output', 'zip', '.', 'careassist_model_output')
print('Downloading careassist_model_output.zip...')
files.download('careassist_model_output.zip')

In [ ]:
# Cell 17: COPY THIS OUTPUT AND PASTE IT BACK IN VS CODE CHAT
print('='*60)
print('COPY EVERYTHING BELOW THIS LINE')
print('='*60)
print(f'MODEL_TYPE={best_name}')
print(f'ROC_AUC={best_auc:.4f}')
print(f'RF_AUC={rf_auc:.4f}')
print(f'XGB_AUC={xgb_auc:.4f}')
print(f'TOTAL_RECORDS={len(df):,}')
print(f'DISRUPTION_RATE={y.mean():.4f}')
print(f'FEATURES={len(feature_names)}')
print(f'TRAIN_SAMPLES={len(X_train_sm):,}')
print(f'TEST_SAMPLES={len(X_test):,}')
print(f'YEARS={sorted(df["FY"].dropna().unique().tolist())}')
print(f'\nTIER_DISTRIBUTION:')
for t in ['Critical','High','Medium','Low']:
    if t in tier_dist.index: print(f'  {t}: {tier_dist[t]:,}')
print(f'\nTOP_10_FEATURES:')
for _,r in feat_imp.head(10).iterrows():
    print(f'  {r["feature"]}: {r["importance"]:.4f}')
print(f'\nCLASSIFICATION_REPORT:')
print(classification_report(y_test, y_pred, target_names=['Stable','Disrupted']))
cm = confusion_matrix(y_test, y_pred)
print(f'CONFUSION_MATRIX: TN={cm[0,0]:,} FP={cm[0,1]:,} FN={cm[1,0]:,} TP={cm[1,1]:,}')
print('='*60)

# CareAssist — Placement Stability Prediction Model
**Train on real AFCARS FY2022–2023 data**

## Instructions
1. Upload your two AFCARS `.tab` files when prompted (FC2022ABv1.tab and FC2023ABv1.tab)
2. Run all cells in order (Runtime → Run all)
3. Download the output files at the end

In [ ]:
# Cell 1: Install packages (shap may take a minute)
!pip install -q xgboost shap imbalanced-learn

In [ ]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)
import xgboost as xgb
import shap
import joblib
from imblearn.over_sampling import SMOTE

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
print('All imports OK ✓')

In [ ]:
# Cell 3: Upload AFCARS data files
from google.colab import files
print('Upload your TWO .tab files: FC2022ABv1.tab and FC2023ABv1.tab')
uploaded = files.upload()

In [ ]:
# Cell 4: Load data (only columns we need)
USE_COLS = [
    'RecNumbr', 'FIPSCode', 'StFCID', 'FY',
    'NUMPLEP', 'TOTALREM', 'CURPLSET', 'CASEGOAL', 'SEX', 'CLINDIS',
    'MR', 'VISHEAR', 'PHYDIS', 'EmotDist', 'OTHERMED', 'CHBEHPRB',
    'PHYABUSE', 'SEXABUSE', 'NEGLECT', 'AAPARENT', 'DAPARENT',
    'AACHILD', 'DACHILD', 'CHILDIS', 'PRTSDIED', 'PRTSJAIL',
    'NOCOPE', 'ABANDMNT', 'RELINQSH', 'HOUSING', 'MANREM',
    'EVERADPT', 'DISREASN', 'PLACEOUT',
    'AgeAtLatRem', 'RaceEthn',
    'SettingLOS', 'LatRemLOS',
]

def load_file(name):
    header = pd.read_csv(name, sep='\t', nrows=0)
    cols = [c for c in USE_COLS if c in header.columns]
    df = pd.read_csv(name, sep='\t', usecols=cols, dtype=str, low_memory=False)
    # Convert numeric
    for c in df.columns:
        if c not in ('RecNumbr', 'FIPSCode', 'StFCID', 'FY'):
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

print('Loading FY2023...')
df23 = load_file('FC2023ABv1.tab')
print(f'  {len(df23):,} records')

print('Loading FY2022...')
df22 = load_file('FC2022ABv1.tab')
print(f'  {len(df22):,} records')

df = pd.concat([df22, df23], ignore_index=True)
print(f'\n📊 Combined: {len(df):,} records × {df.shape[1]} columns')
df.head()

In [ ]:
# Cell 5: Define target variable — placement disruption
# A placement is "disrupted" if:
#   - Discharge reason = 6 (Transfer) or 7 (Runaway)
#   - OR child had 3+ placement settings this episode (instability)
#
# IMPORTANT: We do NOT use num_placements as a feature (it defines the target)

DISRUPTION_CODES = {6, 7}

def label_disruption(row):
    d = row['DISREASN']
    n = row['NUMPLEP']
    if pd.notna(d) and int(d) in DISRUPTION_CODES:
        return 1
    if pd.notna(n) and int(n) >= 3:
        return 1
    return 0

df['disruption'] = df.apply(label_disruption, axis=1)

pos = df['disruption'].sum()
neg = len(df) - pos
print(f'Disrupted (1): {pos:,}  ({100*pos/len(df):.1f}%)')
print(f'Stable    (0): {neg:,}  ({100*neg/len(df):.1f}%)')

# Show disruption by discharge reason
print('\nDisruption by discharge reason:')
print(df.groupby('DISREASN')['disruption'].agg(['count','mean']).to_string())

In [ ]:
# Cell 6: Feature engineering
# NOTE: num_placements (NUMPLEP) is EXCLUDED from features
#        because it's part of the target definition (≥3 = disrupted)

# Age at latest removal
df['age_at_removal'] = df['AgeAtLatRem'].where(df['AgeAtLatRem'] < 99)

# Disability flags
disability_cols = ['MR', 'VISHEAR', 'PHYDIS', 'EmotDist', 'OTHERMED']
for c in disability_cols:
    df[c] = df[c].fillna(0).clip(0, 1).astype(int)
df['has_disability'] = df[disability_cols].max(axis=1)
df['has_clinical_disability'] = (df['CLINDIS'] == 1).astype(int)
df['has_behavioral'] = df['CHBEHPRB'].fillna(0).clip(0, 1).astype(int)

# Removal reason flags
removal_reason_cols = [
    'PHYABUSE', 'SEXABUSE', 'NEGLECT', 'AAPARENT', 'DAPARENT',
    'AACHILD', 'DACHILD', 'CHILDIS', 'PRTSDIED', 'PRTSJAIL',
    'NOCOPE', 'ABANDMNT', 'RELINQSH', 'HOUSING'
]
for c in removal_reason_cols:
    df[c] = df[c].fillna(0).clip(0, 1).astype(int)
df['num_removal_reasons'] = df[removal_reason_cols].sum(axis=1)

# Placement type (categorical)
df['placement_type'] = df['CURPLSET'].where(df['CURPLSET'].isin([1,2,3,4,5,6,7,8]))

# Case goal (categorical)
df['case_goal'] = df['CASEGOAL'].where(df['CASEGOAL'].isin([1,2,3,4,5,6,7]))

# Demographics
df['is_male'] = (df['SEX'] == 1).astype(int)
df['race'] = df['RaceEthn'].where(df['RaceEthn'].isin(range(1, 8)))

# Prior history
df['total_removals'] = df['TOTALREM'].where(df['TOTALREM'] < 98)

# Length-of-stay
df['los_current_setting'] = pd.to_numeric(df.get('SettingLOS', pd.Series(dtype=float)), errors='coerce')
df['los_latest_removal']  = pd.to_numeric(df.get('LatRemLOS', pd.Series(dtype=float)), errors='coerce')

# Other flags
df['ever_adopted'] = df['EVERADPT'].fillna(0).clip(0, 1).astype(int)
df['mandatory_removal'] = df['MANREM'].fillna(0).clip(0, 1).astype(int)

# --- FEATURE LIST (no num_placements to avoid leakage!) ---
FEATURE_COLS = [
    'age_at_removal', 'is_male', 'race',
    'total_removals', 'placement_type', 'case_goal',
    'has_disability', 'has_clinical_disability', 'has_behavioral',
    'num_removal_reasons',
    'PHYABUSE', 'SEXABUSE', 'NEGLECT', 'AAPARENT', 'DAPARENT',
    'NOCOPE', 'ABANDMNT', 'HOUSING',
    'los_current_setting', 'los_latest_removal',
    'ever_adopted', 'mandatory_removal',
]
print(f'{len(FEATURE_COLS)} features selected (num_placements excluded to prevent leakage)')
print(FEATURE_COLS)

In [ ]:
# Cell 7: EDA — Target distribution & key features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Target distribution
df['disruption'].value_counts().plot(kind='bar', ax=axes[0,0], color=['steelblue','coral'])
axes[0,0].set_title('Target: Placement Disruption')
axes[0,0].set_xticklabels(['Stable (0)', 'Disrupted (1)'], rotation=0)

# 2. Age distribution by outcome
df[df['disruption']==0]['age_at_removal'].hist(bins=20, ax=axes[0,1], alpha=0.6, label='Stable', color='steelblue')
df[df['disruption']==1]['age_at_removal'].hist(bins=20, ax=axes[0,1], alpha=0.6, label='Disrupted', color='coral')
axes[0,1].set_title('Age at Removal by Outcome')
axes[0,1].legend()

# 3. Disruption rate by placement type
pt_labels = {1:'Pre-Adopt',2:'Foster-Rel',3:'Foster-NonRel',4:'Group Home',5:'Institution',6:'Sup IL',7:'Runaway',8:'Trial Home'}
pt_rates = df.groupby('placement_type')['disruption'].mean().sort_values(ascending=False)
pt_rates.index = [pt_labels.get(int(i), str(i)) for i in pt_rates.index]
pt_rates.plot(kind='barh', ax=axes[0,2], color='steelblue')
axes[0,2].set_title('Disruption Rate by Placement Type')
axes[0,2].set_xlabel('Disruption Rate')

# 4. Disruption rate by case goal
cg_labels = {1:'Reunify',2:'Relative',3:'Adoption',4:'Long-term FC',5:'Emancipation',6:'Guardianship',7:'Not established'}
cg_rates = df.groupby('case_goal')['disruption'].mean().sort_values(ascending=False)
cg_rates.index = [cg_labels.get(int(i), str(i)) for i in cg_rates.index]
cg_rates.plot(kind='barh', ax=axes[1,0], color='coral')
axes[1,0].set_title('Disruption Rate by Case Goal')
axes[1,0].set_xlabel('Disruption Rate')

# 5. Disruption rate by total removals
tr_rates = df.groupby('total_removals')['disruption'].mean().head(10)
tr_rates.plot(kind='bar', ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Disruption Rate by # Total Removals')
axes[1,1].tick_params(axis='x', rotation=0)

# 6. Removal reason correlation with disruption
rr_corr = df[removal_reason_cols + ['disruption']].corr()['disruption'].drop('disruption').sort_values()
rr_corr.plot(kind='barh', ax=axes[1,2], color='coral')
axes[1,2].set_title('Removal Reason Correlation w/ Disruption')

plt.suptitle('CareAssist — Exploratory Data Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved ✓')

In [ ]:
# Cell 8: Prepare modelling data
model_df = df[FEATURE_COLS + ['disruption']].copy()

# One-hot encode categoricals
cat_cols = ['placement_type', 'case_goal', 'race']
model_df = pd.get_dummies(model_df, columns=cat_cols, prefix=cat_cols, dummy_na=False)

# Fill NaN with median
for c in model_df.columns:
    if model_df[c].isna().any():
        model_df[c] = model_df[c].fillna(model_df[c].median())

y = model_df['disruption']
X = model_df.drop(columns=['disruption'])
feature_names = list(X.columns)

print(f'Features: {len(feature_names)}')
print(f'Records:  {len(X):,}')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Disruption rate — Train: {y_train.mean():.3f}  Test: {y_test.mean():.3f}')

# SMOTE for class imbalance
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
print(f'After SMOTE: {len(X_train_sm):,} training samples (balanced)')

In [ ]:
# Cell 9: Train Random Forest
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sm, y_train_sm)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_proba)
rf_ap  = average_precision_score(y_test, rf_proba)
print(f'  ROC-AUC:          {rf_auc:.4f}')
print(f'  Avg Precision:    {rf_ap:.4f}')

In [ ]:
# Cell 10: Train XGBoost
print('Training XGBoost...')
scale_pos = (y_train_sm == 0).sum() / max((y_train_sm == 1).sum(), 1)
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
)
xgb_model.fit(X_train_sm, y_train_sm, verbose=False)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_proba)
xgb_ap  = average_precision_score(y_test, xgb_proba)
print(f'  ROC-AUC:          {xgb_auc:.4f}')
print(f'  Avg Precision:    {xgb_ap:.4f}')

# Pick best model
if xgb_auc > rf_auc:
    best_model, best_proba, best_name, best_auc = xgb_model, xgb_proba, 'XGBoost', xgb_auc
else:
    best_model, best_proba, best_name, best_auc = rf, rf_proba, 'RandomForest', rf_auc

print(f'\n✅ Best model: {best_name} (AUC = {best_auc:.4f})')

In [ ]:
# Cell 11: Model Evaluation — ROC, PR curves, Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ROC Curve (both models)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_xg, tpr_xg, _ = roc_curve(y_test, xgb_proba)
axes[0].plot(fpr_rf, tpr_rf, 'b-', lw=2, label=f'Random Forest AUC={rf_auc:.3f}')
axes[0].plot(fpr_xg, tpr_xg, 'r-', lw=2, label=f'XGBoost AUC={xgb_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--',lw=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Precision-Recall Curve
prec_rf, rec_rf, _ = precision_recall_curve(y_test, rf_proba)
prec_xg, rec_xg, _ = precision_recall_curve(y_test, xgb_proba)
axes[1].plot(rec_rf, prec_rf, 'b-', lw=2, label=f'RF AP={rf_ap:.3f}')
axes[1].plot(rec_xg, prec_xg, 'r-', lw=2, label=f'XGB AP={xgb_ap:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

# Confusion Matrix
y_pred = (best_proba >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=['Stable', 'Disrupted'],
            yticklabels=['Stable', 'Disrupted'], ax=axes[2])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title(f'Confusion Matrix ({best_name})')

plt.suptitle('CareAssist — Model Evaluation', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Stable', 'Disrupted']))

In [ ]:
# Cell 12: Feature Importance
importances = best_model.feature_importances_
feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top20 = feat_imp.head(20)
ax.barh(range(20), top20['importance'].values, color='steelblue')
ax.set_yticks(range(20))
ax.set_yticklabels(top20['feature'].values)
ax.invert_yaxis()
ax.set_title(f'Top 20 Feature Importances ({best_name})')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 20 Features:')
print(feat_imp.head(20).to_string(index=False))

In [ ]:
# Cell 13: SHAP Analysis
print('Computing SHAP values (sample of 1000)...')
explainer = shap.TreeExplainer(best_model)
X_sample = X_test.sample(min(1000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)

# For binary classifiers, shap_values may be a list [class0, class1]
if isinstance(shap_values, list):
    shap_values = shap_values[1]

# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                  show=False, max_display=20)
plt.title('SHAP Feature Importance')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Waterfall for a single high-risk case
high_risk_idx = X_sample.iloc[np.argmax(best_model.predict_proba(X_sample)[:,1])]
idx_pos = np.argmax(best_model.predict_proba(X_sample)[:,1])
shap.plots.waterfall(shap.Explanation(
    values=shap_values[idx_pos],
    base_values=explainer.expected_value if not isinstance(explainer.expected_value, list) else explainer.expected_value[1],
    data=X_sample.iloc[idx_pos],
    feature_names=feature_names
), show=True)
print('SHAP analysis complete ✓')

In [ ]:
# Cell 14: Score all records & create risk tiers
print('Scoring all records...')
X_all = model_df.drop(columns=['disruption'])
all_proba = best_model.predict_proba(X_all)[:, 1]
df['priority_score'] = all_proba

# Risk tiers
df['risk_tier'] = pd.cut(
    df['priority_score'],
    bins=[0, 0.3, 0.6, 0.8, 1.0],
    labels=['Low', 'Medium', 'High', 'Critical'],
    include_lowest=True
)

print('\n📊 Risk Tier Distribution:')
tier_dist = df['risk_tier'].value_counts()
for tier in ['Critical', 'High', 'Medium', 'Low']:
    if tier in tier_dist.index:
        cnt = tier_dist[tier]
        print(f'  {tier:>10}: {cnt:>8,}  ({100*cnt/len(df):.1f}%)')

# Distribution plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['priority_score'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0.3, color='green', linestyle='--', label='Low/Medium (0.3)')
ax.axvline(0.6, color='orange', linestyle='--', label='Medium/High (0.6)')
ax.axvline(0.8, color='red', linestyle='--', label='High/Critical (0.8)')
ax.set_xlabel('Priority Score (Disruption Probability)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Priority Scores')
ax.legend()
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 15: Export everything
os.makedirs('careassist_model_output', exist_ok=True)

# 1. Save trained model
joblib.dump(best_model, 'careassist_model_output/placement_model.pkl')
print('✓ Model saved: placement_model.pkl')

# 2. Save model metadata
meta = {
    'model_type': best_name,
    'roc_auc': round(best_auc, 4),
    'rf_auc': round(rf_auc, 4),
    'xgb_auc': round(xgb_auc, 4),
    'feature_names': feature_names,
    'feature_count': len(feature_names),
    'train_samples': int(len(X_train_sm)),
    'test_samples': int(len(X_test)),
    'total_records': int(len(df)),
    'disruption_rate': round(float(y.mean()), 4),
    'threshold': 0.5,
    'risk_tiers': {'Low': '0-0.3', 'Medium': '0.3-0.6', 'High': '0.6-0.8', 'Critical': '0.8-1.0'}
}
with open('careassist_model_output/model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('✓ Metadata saved: model_metadata.json')

# 3. Save feature importance
feat_imp.to_csv('careassist_model_output/feature_importance.csv', index=False)
print('✓ Feature importance saved: feature_importance.csv')

# 4. Save scored cases
export_cols = [
    'RecNumbr', 'StFCID', 'FIPSCode', 'FY',
    'age_at_removal', 'is_male', 'race',
    'total_removals', 'placement_type', 'case_goal',
    'has_disability', 'has_clinical_disability', 'has_behavioral',
    'num_removal_reasons',
    'los_current_setting', 'los_latest_removal',
    'disruption', 'priority_score', 'risk_tier',
]
export_cols = [c for c in export_cols if c in df.columns]
scored_df = df[export_cols].copy()
scored_df.to_csv('careassist_model_output/scored_cases.csv', index=False)
print(f'✓ Scored cases saved: scored_cases.csv ({len(scored_df):,} records)')

# 5. Copy plots
import shutil
for f in ['eda_plots.png', 'model_evaluation.png', 'feature_importance.png', 'shap_summary.png', 'score_distribution.png']:
    if os.path.exists(f):
        shutil.copy(f, f'careassist_model_output/{f}')

print('\n📁 All outputs in: careassist_model_output/')
print('Files:', os.listdir('careassist_model_output'))

In [ ]:
# Cell 16: Download all outputs as a zip
import shutil
shutil.make_archive('careassist_model_output', 'zip', '.', 'careassist_model_output')
print('\n📦 Created: careassist_model_output.zip')
print('\nDownloading...')
files.download('careassist_model_output.zip')

In [ ]:
# Cell 17: Print summary for copy-paste back to VS Code
print('='*60)
print('COPY EVERYTHING BELOW THIS LINE BACK TO THE CHAT')
print('='*60)
print(f'MODEL_TYPE={best_name}')
print(f'ROC_AUC={best_auc:.4f}')
print(f'RF_AUC={rf_auc:.4f}')
print(f'XGB_AUC={xgb_auc:.4f}')
print(f'TOTAL_RECORDS={len(df):,}')
print(f'DISRUPTION_RATE={y.mean():.4f}')
print(f'FEATURES={len(feature_names)}')
print(f'TRAIN_SAMPLES={len(X_train_sm):,}')
print(f'TEST_SAMPLES={len(X_test):,}')
print(f'\nTIER_DISTRIBUTION:')
for tier in ['Critical', 'High', 'Medium', 'Low']:
    if tier in tier_dist.index:
        print(f'  {tier}: {tier_dist[tier]:,}')
print(f'\nTOP_10_FEATURES:')
for _, r in feat_imp.head(10).iterrows():
    print(f'  {r["feature"]}: {r["importance"]:.4f}')
print(f'\nCLASSIFICATION_REPORT:')
print(classification_report(y_test, y_pred, target_names=['Stable', 'Disrupted']))
cm = confusion_matrix(y_test, y_pred)
print(f'CONFUSION_MATRIX: TN={cm[0,0]:,} FP={cm[0,1]:,} FN={cm[1,0]:,} TP={cm[1,1]:,}')
print('='*60)